# Agent: news

Develop and test **`agentic_scd.agents.news.news_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    U["new_signals<br/>DisruptionSignal[]"]:::faded --> N1
    subgraph N["news_node (per signal)"]
        N1["title + raw_text + region"] --> N2{"real LLM enabled?"}
        N2 -->|yes| N3["Groq extraction to JSON"]
        N2 -->|no or invalid| N4["heuristic event/entity extraction"]
        N3 --> N5["normalize event_type, entities, region, summary"]
        N4 --> N5
    end
    N5 --> D["event_analyses<br/>EventAnalysis[]"]
    D --> DOWN["downstream: weather · classify"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```


**State contract**

- **Reads:** `new_signals` (list of `DisruptionSignal`)
- **Writes:** `event_analyses` (list of `EventAnalysis`: event_type, entities, extracted_region, severity_hint, summary)
- **Fallback / degradation:** if the real LLM is disabled or returns invalid JSON, the node falls back to deterministic heuristic extraction

This notebook exercises the same shared `news_node` used by the graph builder, so the notebook path and the runtime path stay aligned.

## Is the DB up? (optional)

In [ ]:
from agentic_scd.config import get_settings
from agentic_scd.devtools import db_status

status = db_status()
settings = get_settings()
print(status.detail)
print(f"llm_is_mock={settings.llm_is_mock} model={settings.groq_model}")
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")


## Build a representative input state

In [ ]:
from agentic_scd.devtools import sample_state

state = sample_state(count=2)
print("Input new_signals:")
for signal in state["new_signals"]:
    print(" -", signal.title)


## Call `analyze_signal` on one record

In [ ]:
from agentic_scd.agents.news import analyze_signal

analysis = analyze_signal(state["new_signals"][0])
print("event_type:", analysis.event_type)
print("entities:", ", ".join(analysis.entities) or "-")
print("region:", analysis.extracted_region or "-")
print("severity_hint:", analysis.severity_hint or "-")
print("summary:", analysis.summary)


## Call `news_node` in isolation

In [ ]:
from agentic_scd.agents.news import news_node

state.update(news_node(state))
for item in state["event_analyses"]:
    entities = ", ".join(item.entities) or "-"
    print(f"{item.event_type:>24}  region={item.extracted_region or '-':<16} entities={entities}")


## Iterate here

This is your dev surface: tweak the sample signals above, switch real/mock LLM mode through the environment, and re-run the cells. When you want to see the full handoff, open `00_orchestration` and stream the graph with the same backend package.